# Company Peer Benchmark

Select a company and benchmark technology adoption and profitability against its industry peers

In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

df = pd.read_csv("data/software_companies_dataset_v2.csv")

# Prepare missing values for reliable widgets and Plotly charts.
categorical_columns = [
    "Company_Name", "Industry", "Headquarters_City", "Country",
    "Ownership_Type", "Customer_Segment", "Primary_Cloud", "Risk_Rating"
]
for column in categorical_columns:
    if column in df.columns:
        df[column] = df[column].fillna("Unknown").astype(str).str.strip()

numeric_columns = [
    "Employees", "Annual_Revenue", "Profit_Margin", "Market_Share",
    "R&D_Spending", "Average_Salary", "Training_Hours_Per_Employee",
    "Employee_Satisfaction", "Adoption_Rate_AI", "Adoption_Rate_Cloud",
    "Adoption_Rate_Blockchain"
]
for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")
        df[column] = df[column].fillna(df[column].median())

df["Annual_Revenue"] = df["Annual_Revenue"].clip(lower=1)
df["Employees"] = df["Employees"].clip(lower=1)

df["Technology_Index"] = df[
    ["Adoption_Rate_AI","Adoption_Rate_Cloud","Adoption_Rate_Blockchain"]].mean(axis=1)
company = widgets.Dropdown(options=sorted(df["Company_Name"].dropna().astype(str).unique().tolist()), description="Company")
out = widgets.Output()

def render(*_):
    row = df[df["Company_Name"] == company.value].iloc[0]
    peers = df[df["Industry"] == row["Industry"]]
    metrics = ["Adoption_Rate_AI","Adoption_Rate_Cloud","Adoption_Rate_Blockchain"]
    comp = pd.DataFrame({
        "Metric":["AI","Cloud","Blockchain"],
        "Company":[row[m] for m in metrics],
        "Peer median":[peers[m].median() for m in metrics]
    }).melt("Metric", var_name="Series", value_name="Value")
    with out:
        clear_output(wait=True)
        display(widgets.HTML(f"<h3>{company.value} — peer group: {row['Industry']}</h3>"))
        px.bar(comp, x="Metric", y="Value", color="Series", barmode="group",
               range_y=[0,100], title="Technology adoption benchmark").show()
        px.scatter(peers, x="Technology_Index", y="Profit_Margin",
                   size="Annual_Revenue", hover_name="Company_Name",
                   title="Peer landscape").show()

company.observe(render, names="value")
display(company, out)
render()